
**Created by:**

__Viktor Varga__

<br>
<img src="../_files/nb_im/ik_logo.png" style="display:inline-block" width='40%'>
<hr>

# **NumPy Array Programming Guide** – Chapter 4


## Referencing array elements – Integer index arrays (Advanced indexing 1.)


In Chapter 2 we saw that the *basic indexing* syntax makes it possible to reference multiple elements of an array at once. With this approach, we could reference groups of elements whose indices along each axis came from given intervals, produced with fixed step sizes.

This approach is advantageous because if we select elements from an array in this way for further operations, there is no need to physically copy them into a new array: by adjusting the iterators present in the array implementation, we can easily reference those elements as an array *view*. The drawback of *basic indexing* is that the iterators must step through the array with fixed strides. If the elements we want to reference are not laid out at regular strides in the array, we cannot define a view that references exactly those elements.

NumPy fortunately also allows referencing multiple elements at once by providing the exact sequence of indices, via the **advanced indexing** syntax. NumPy supports two variants of *advanced indexing*: in this chapter we cover element selection using sequences/arrays of integer indices, while the next chapter covers selecting elements using a mask of boolean values.

Below we demonstrate the *advanced indexing* syntax through examples. First, let’s create a one‑dimensional array!


In [ ]:
import numpy as np

a = np.arange(0, 100, 10, dtype=np.int32)

print("The 'a' array:", a)

The 'a' array: [ 0 10 20 30 40 50 60 70 80 90]


If `a` were a Python list, we would need a loop (or an equivalent list comprehension) to select multiple arbitrary elements of the list for further computation, for example to sum the chosen elements.

In NumPy, this is not necessary. Create another one‑dimensional integer array (here called `idx_arr`) that stores the indices of the elements we want to reference. Indices can of course repeat; they do not have to be all different.

As shown below, the expression `a[idx_arr]` returns the elements at the desired indices placed into a new array. If we want to provide the indices manually, we do not need to wrap them into an array - using a list in place of `idx_arr` also works, and NumPy will automatically convert it to an array:


In [ ]:
idx_arr = np.array([3,5,8,1,8,5])

print("Indexing with int array -> a[idx_arr]:", a[idx_arr])

print("Indexing with list -> a[[3,5,8,1,8,5]]:", a[[3,5,8,1,8,5]])

# print("Indexing with a tuple is INCORRECT this way -> a[(3,5,8,1,8,5)]:", a[(3,5,8,1,8,5)])     # IndexError: too many indices for array: array is 1-dimensional, but 6 were indexed


Indexing with int array -> a[idx_arr]: [30 50 80 10 80 50]
Indexing with list -> a[[3,5,8,1,8,5]]: [30 50 80 10 80 50]


Just as when referencing individual elements, negative indices can also be used here.


In [ ]:
print("a[[-1,-3,-5,-8]]:", a[[-1,-3,-5,-8]])    # OK

# print("a[[-12, -9]]:", a[[-12, -9]])    # IndexError: index -12 is out of bounds for axis 0 with size 10

# print("a[[12, 9]]:", a[[12, 9]])          # IndexError: index 12 is out of bounds for axis 0 with size 10

a[[-1,-3,-5,-8]]: [ 9  7 99 77]


If *advanced indexing* appears on the left-hand side of an assignment, the referenced elements of the array can be overwritten:


In [ ]:
a = np.arange(10, dtype=np.int32)
print("The 'a' array:", a)

a[[5,6,2]] = [99,42,77]
# a[[5,6,2]] = 99,42,77                                 # SAME as above
# a[[5,6,2]] = (99,42,77)                               # SAME as above
# a[[5,6,2]] = np.array([99,42,77], dtype=np.int32)     # SAME as above

print("The modified 'a' array:", a)

The 'a' array: [0 1 2 3 4 5 6 7 8 9]
The modified 'a' array: [ 0  1 77  3  4 99 42  7  8  9]


An important difference compared to *basic indexing* is that while, for example, `a[2:5]` is a view of `a`, `a[[2,5]]` is a newly allocated two‑element array into which `a[2]` and `a[5]` are copied. Accordingly, if we modify a view obtained via *basic indexing*, the original array also changes, but **if we modify the elements of the new array created via *advanced indexing*, the original array does not change.**


In [ ]:
a = np.arange(10, dtype=np.int32)
print("The 'a' array:", a)

a2 = a[[5,6,2]]
print("The 'a2' array:", a2)

a2[0] = 42
print("The modified 'a2' array:", a2)

print("The 'a' array after 'a2' was modified:", a)   # unchanged

The 'a' array: [0 1 2 3 4 5 6 7 8 9]
The 'a2' array: [5 6 2]
The modified 'a2' array: [42  6  2]
The 'a' array after 'a2' was modified: [0 1 2 3 4 5 6 7 8 9]


Since the indexing array lists the indices of the elements to be referenced, **the shape of the array returned by *advanced indexing* matches the shape of the index array**, as long as the indexed array itself is one‑dimensional.
<br>$\ $</br>

<img src='../_files/nb_im/numpy_basics-adv_idx_br0_1d.png' width='60%'>

As you can also see at the end of the code block below, you need to be careful when using structures that stand in for multidimensional index arrays, such as nested lists. In the example below, NumPy interprets the nested list of depth two as if we were indexing the array along two axes with two separate simple lists.


In [ ]:
a = np.arange(10, dtype=np.int32)
print("The 'a' array:", a)
print("a.shape:", a.shape)

my_idxs = np.array([[3,7,-2], [6,-4,0]], dtype=np.int32)
print("\nmy_idxs:\n", my_idxs)
print("my_idxs.shape:", my_idxs.shape)

r = a[my_idxs]
print("\na[my_idxs]:\n", r)
print("a[my_idxs].shape:", r.shape)

# Indexing directly with nested lists is not possible this way - interpreted as indexing over multiple axes:
#
# print("a[ [[3,7,-2], [6,-8,0]] ]:", a[[[3,7,-2], [6,-8,0]]])    # IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed



The 'a' array: [0 1 2 3 4 5 6 7 8 9]
a.shape: (10,)

my_idxs:
 [[ 3  7 -2]
 [ 6 -4  0]]
my_idxs.shape: (2, 3)

a[my_idxs]:
 [[3 7 8]
 [6 6 0]]
a[my_idxs].shape: (2, 3)


Naturally, *advanced indexing* can also be applied to multidimensional arrays. A simple case is shown in the cell below:


In [ ]:
a = np.arange(6, dtype=np.int32).reshape(2,3)
print("The 'a' array:\n", a)
print("a.shape:", a.shape)

print("\na[[1,1], [0,2]]:", a[[1,1,0], [0,2,2]])   # indexing three items: [1,0], [1,2] and [0,2]
print("a[ ([1,1], [0,2]) ]:", a[([1,1,0], [0,2,2])])    # equivalent to line above


The 'a' array:
 [[0 1 2]
 [3 4 5]]
a.shape: (2, 3)

a[[1,1], [0,2]]: [3 5 2]
a[ ([1,1], [0,2]) ]: [3 5 2]


In the cell above we have a two‑dimensional array of shape (2, 3). We want to reference three elements at once: the elements at indices [1, 0], [1, 2], and [0, 2], in that order. To do this, we separate the indices along axis #0 and axis #1, and place them into separate arrays/lists. We can then use these two index arrays/lists to index along the two axes. The three elements can be referenced at once with the expression `a[[1,1,0], [0,2,2]]`. As mentioned, the result is a new array that contains the values of the three selected elements.

Of course, we can also use multidimensional arrays to index multidimensional arrays. The easiest case to understand is when the index arrays used along each axis have the same shape. In that case, the shape of the resulting array containing the referenced elements will be the same:
<br>$\ $</br>

<img src='../_files/nb_im/numpy_basics-adv_idx_br0.png' width='70%'>

NumPy is fortunately more permissive than this: **the arrays used for simultaneous indexing along different axes do not have to have identical shapes. In fact, it is sufficient if their shapes can be made compatible via the *broadcasting* mechanism.**


When indexing a multidimensional array, we can mix methods along different axes: reference a single element, slice using *basic indexing*, or reference multiple elements using *advanced indexing*:
<br>$\ $</br>

<img src='../_files/nb_im/numpy_basics-adv_idx.png' width='60%'>

It is important to note that **if we use *advanced indexing* along *any* axis, the result containing the referenced elements is a new array, not a view** of the original indexed array.


In [ ]:
r1 = a[[1,0,0], :]
print("a[[1,0,0],:]:\n", r1)
print("\na[[1,0,0],:].shape:", r1.shape)

r2 = a[[0,-1], 2::-2]
print("\na[[0,-1], 2::-2]]:\n", r2)
print("\na[[0,-1], 2::-2]].shape:", r2.shape)

r3 = a[0, [1]]
print("\na[0, [1]]:", r3)
print("a[0, [1]].shape:", r3.shape)

r1[0] = 42
r2[0] = 42
r3[0] = 42

print("\nThe 'a' array after modifying 'r1', 'r2', 'r3':\n", a)  # 'a' array is unchanged
                                                                 #     as r1, r2, r3 are all copies


a[[1,0,0],:]:
 [[3 4 5]
 [0 1 2]
 [0 1 2]]

a[[1,0,0],:].shape: (3, 3)

a[[0,-1], 2::-2]]:
 [[2 0]
 [5 3]]

a[[0,-1], 2::-2]].shape: (2, 2)

a[0, [1]]: [1]
a[0, [1]].shape: (1,)

The 'a' array after modifying 'r1', 'r2', 'r3':
 [[0 1 2]
 [3 4 5]]


With the *advanced indexing* syntax we can of course reference a single element as well. In that case, the result is placed into a new array. While `a[0, 2]` returns a scalar (since we reference a single element with scalar indices), `a[[0], [2]]`—or even `a[0, [2]]` and `a[[0], 2]`—returns a one‑element array in all three cases, because we use *advanced indexing* along at least one axis:


In [ ]:
r4 = a[0,2]
print("\na[0,2]:", r4)
print("type(a[0,2]):", type(r4))

r5 = a[0,[2]]                      # equivalent to a[ [0], [2] ]
print("\na[0,[2]]:", r5)
print("type(a[0,[2]]):", type(r5))
print("a[0,[2]].shape:", r5.shape)



a[0,2]: 2
type(a[0,2]): <class 'numpy.int32'>

a[0,[2]]: [2]
type(a[0,[2]]): <class 'numpy.ndarray'>
a[0,[2]].shape: (1,)


A typical use case of *advanced indexing* is when we want to reference exactly one, pre‑specified element from each row/column of an array—and the index can differ from row to row or column to column.

For example, to construct a so‑called “one‑hot” encoded array often used with neural networks, we need exactly this. In a 2D array, we want to set one element in each row to 1, while the rest of the elements in that row are 0. Let `idxs = [4, 0, 1, 1]` specify which element should be set to 1 in each of the four rows. Let the length of each one‑hot vector (row) be 5 this time. How can we construct the desired one‑hot array using *advanced indexing*?

Based on the description above, the result array has 4 rows and 5 columns, i.e. shape (4, 5). To write the ones, we must index both axes simultaneously, using two lists/arrays of length 4 (because we write four elements—one per row). The target indices are: `[0, 4], [1, 0], [2, 1], [3, 1]`. That is, along the first axis we use `[0, 1, 2, 3]`, and along the second axis we use `[4, 0, 1, 1]`. More generally, along the first axis we step through the rows, so `np.arange(n_rows)` generates those indices, and for the second axis we can use the `idxs` list itself:


In [ ]:
idxs = [4, 0, 1, 1]
one_hot = np.zeros((4, 5), dtype=np.float32)
one_hot[np.arange(one_hot.shape[0]), idxs] = 1.

print("The 'one_hot' array:\n", one_hot)

The 'one_hot' array:
 [[0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0.]]


With the same technique we can of course also *read* similarly arranged elements from arrays; it is not limited to writing.

The technique can be extended: we can enumerate indices from 0 upward along multiple axes. For example, it can be used to reference a particular component of same‑size vectors placed at points of an N‑dimensional grid. For this, we either need to manually align the `np.arange()` index arrays used along multiple axes with appropriate (multiplication-table-style / outer-product-style) broadcasting, or we can use the `np.meshgrid()` function.


## Concatenating arrays, inserting elements


Occasionally, the data we want to work with comes from different sources, or different parts of it are produced by different operations. In computer vision, a common problem is loading images of different resolutions, scaling them to a common resolution, and then storing them in a single array. In such cases we can allocate a sufficiently large array and then write each source into the appropriate region, using individual statements.

To make these steps easier, NumPy provides ways to **join different arrays together** and thereby build larger arrays. Because, unlike lists, the size of arrays cannot be modified in place, when we join arrays the background operation is exactly what you would do manually: a new array of appropriate size is allocated and the different input arrays are copied into it. In the program code, however, using the `np.concatenate()` and `np.stack()` functions, this operation can be performed in a single instruction.

`np.concatenate()` can combine multiple arrays into a new, larger array, provided that they have the same number of dimensions and their shapes differ along at most one axis. This axis will be the one along which we concatenate the arrays. Below, we combine an array of shape (5,) and an array of shape (3,) into a new array of shape (8,):




In [ ]:
arr1 = np.arange(5, dtype=np.int32)
arr2 = np.arange(3, dtype=np.int32)

print("np.concatenate([arr1, arr2]):", np.concatenate([arr1, arr2]))

np.concatenate([arr1, arr2]): [0 1 2 3 4 0 1 2]


As mentioned above, for multidimensional arrays the input arrays must have the same length along every axis except one: the axis along which we perform concatenation. The input arrays can, for example, be listed in a Python list. With the `axis` parameter we specify along which axis we want to concatenate the arrays:
<br>$\ $</br>

<img src='../_files/nb_im/numpy_basics-concat.png' width='60%'>

If the arrays have the same shape, the `axis` parameter can be chosen from multiple axes.


In [ ]:
arr1 = np.zeros((2,1,2), dtype=np.int32)
arr2 = np.ones((2,3,2), dtype=np.float64)
arr3 = np.full((2,1,2), dtype=np.uint8, fill_value=42)

cat_arr = np.concatenate([arr1, arr2, arr3], axis=1)

print("np.concatenate([arr1, arr2, arr3], axis=1):\n", cat_arr)
print("    and its shape is:", cat_arr.shape)

np.concatenate([arr1, arr2, arr3], axis=1):
 [[[ 0.  0.]
  [ 1.  1.]
  [ 1.  1.]
  [ 1.  1.]
  [42. 42.]]

 [[ 0.  0.]
  [ 1.  1.]
  [ 1.  1.]
  [ 1.  1.]
  [42. 42.]]]
    and its shape is: (2, 5, 2)


Along the concatenation axis, the individual arrays may have arbitrary lengths - even arrays with length 0 (empty arrays) can be concatenated. Naturally, empty arrays do not change the result, but `np.concatenate()` does not raise an error for such inputs.


Besides `np.concatenate()`, `np.stack()` provides similar functionality. However, **while `np.concatenate()` joins the input arrays along an *existing* axis, `np.stack()` always creates a *new* axis and lists the arrays along that axis** in an array with one additional dimension.

Therefore, the inputs to `np.stack()` must have exactly the same shape along every axis, and the length of the newly created axis equals the number of input arrays.

As shown in the figure and code cell below, stacking three arrays of shape (1, 2) along axis #0 produces shape (3, 1, 2); along axis #1 it produces shape (1, 3, 2); and along axis #2 it produces shape (1, 2, 3).
<br>$\ $</br>

<img src='../_files/nb_im/numpy_basics-stack.png' width='70%'>


In [ ]:
arr1 = np.zeros((2,1), dtype=np.int32)
arr2 = np.ones((2,1), dtype=np.float64)
arr3 = np.full((2,1), dtype=np.uint8, fill_value=99)

stacked1 = np.stack([arr1, arr2, arr3], axis=0)
print("np.stack([arr1, arr2, arr3], axis=0):\n", stacked1)
print("    and its shape is:", stacked1.shape)

stacked2 = np.stack([arr1, arr2, arr3], axis=1)
print("\nnp.stack([arr1, arr2, arr3], axis=1):\n", stacked2)
print("    and its shape is:", stacked2.shape)

stacked3 = np.stack([arr1, arr2, arr3], axis=2)
print("\nnp.stack([arr1, arr2, arr3], axis=2):\n", stacked3)
print("    and its shape is:", stacked3.shape)

np.stack([arr1, arr2, arr3], axis=0):
 [[[ 0.]
  [ 0.]]

 [[ 1.]
  [ 1.]]

 [[99.]
  [99.]]]
    and its shape is: (3, 2, 1)

np.stack([arr1, arr2, arr3], axis=1):
 [[[ 0.]
  [ 1.]
  [99.]]

 [[ 0.]
  [ 1.]
  [99.]]]
    and its shape is: (2, 3, 1)

np.stack([arr1, arr2, arr3], axis=2):
 [[[ 0.  1. 99.]]

 [[ 0.  1. 99.]]]
    and its shape is: (2, 1, 3)


In the cell above, the statement
```
stacked1 = np.stack([arr1, arr2, arr3], axis=0)
```
is equivalent to
```
stacked1 = np.concatenate([arr1[None,:,:], arr2[None,:,:], arr3[None,:,:]], axis=0)
```

The shapes of the input arrays must strictly satisfy the rules described above. **Neither `np.concatenate()` nor `np.stack()` supports automatic shape alignment (*broadcasting*).** If we want to align shapes before concatenation, we need explicit alignment, which can be achieved by calling `np.broadcast_to()` on the appropriate input arrays.

We can also extend arrays by one or more elements. Using `np.append()`, we can insert new elements/array slices at the end of the axes we choose. Using the `np.insert()` function, elements or slices can be inserted along any axis, not only at the end of the axes, but also after any index or at the beginning of the axes. The `np.pad()` function supplements an array in a "frame" manner: it inserts a specified number of new elements at the beginning and end of each axis.

_**Note:** Arrays returned by `np.concatenate()` and `np.stack()` always allocate new memory space large enough to hold all elements of all input arrays. Similarly, the `np.pad()`, `np.append()`, and `np.insert()` functions also create new arrays, even if we only insert a single element into our array, since, as we stated earlier, NumPy arrays are not resizable. For this reason, **using NumPy arrays as lists, with repeated insertions in a loop or concatenation, is an incorrect technique and a bad programming pattern.** If we create parts of a larger array in independent steps and know the size of this larger array in advance, it is worth reserving an empty array and then writing the individual parts into its corresponding slices one by one. If we do not know the size of the larger array in advance, it is better to collect the completed subarrays in a Python list, and then concatenate the contents of the list into the final, larger array at the very end!_


## Sorting array elements, logarithmic-time search


There are several reasons to sort the elements of an array. We may want to find the largest or smallest elements but not be satisfied with only the maximum or minimum value; we might need the full ordering.

To sort the elements of an array we can use `np.sort()`:


In [ ]:
a = np.array([5,5,3,2,4,5,0,4], dtype=np.int32)
print("The 'a' array:", a)

sa = np.sort(a)
print("The sorted 'sa' array:", sa)
print("The sorted (descending) 'sa' array:", sa[::-1])

The 'a' array: [5 5 3 2 4 5 0 4]
The sorted 'sa' array: [0 2 3 4 4 5 5 5]
The sorted (descending) 'sa' array: [5 5 5 4 4 3 2 0]


If we want to reuse the ordering induced by sorting for the sorting of another array, we can use `np.argsort()`. This function does not return the sorted array itself; it returns the index array such that indexing the original array with it yields the sorted array:


In [ ]:
labels = np.array([5,5,3,2,4,5,0,4], dtype=np.int32)
values = np.array([0.2, 0.8, 1.3, -0.5, 2.7, -0.1, 3., 1.1], dtype=np.float32)

print("The 'labels' array:", labels)
print("The 'values' array:", values)
print("Their shape:", labels.shape, values.shape)

value_sorter = np.argsort(values)
print("\nThe indices which sort the 'values' array:", value_sorter)

labels_sorted_by_values = labels[value_sorter]
print("The 'labels' array sorted by values in the 'values' array:", labels_sorted_by_values)

print("\n'values[value_sorter]' returns the sorted 'values' array:", values[value_sorter])

The 'labels' array: [5 5 3 2 4 5 0 4]
The 'values' array: [ 0.2  0.8  1.3 -0.5  2.7 -0.1  3.   1.1]
Their shape: (8,) (8,)

The indices which sort the 'values' array: [3 5 0 1 7 2 4 6]
The 'labels' array sorted by values in the 'values' array: [2 5 5 5 4 3 4 0]

'values[value_sorter]' returns the sorted 'values' array: [-0.5 -0.1  0.2  0.8  1.1  1.3  2.7  3. ]


We can also sort slices of a multidimensional array independently along a specified axis. For this, `np.sort()` and `np.argsort()` accept an `axis` parameter:


In [ ]:
labels2d = labels.reshape((2, -1))
print("The 'labels2d' array:\n", labels2d)

print("\nThe 'labels2d' array sorted along axis#0:\n", np.sort(labels2d, axis=0))
print("The 'labels2d' array sorted along axis#1:\n", np.sort(labels2d, axis=1))

print("\nSorters of the 'labels2d' array along axis#0:\n", np.argsort(labels2d, axis=0))
print("Sorters of the 'labels2d' array along axis#1:\n", np.argsort(labels2d, axis=1))


print("\n -- Sorting the 'labels2d' array along axis#1 by indexing it with the sorters:")

sorter_ax1 = np.argsort(labels2d, axis=1)
sorted_labels2d_ax1 = labels2d[np.arange(labels2d.shape[0])[:,None], sorter_ax1]    # broadcasting the index arrays of the two axes: (2,1) vs (2,4)
print(sorted_labels2d_ax1)


The 'labels2d' array:
 [[5 5 3 2]
 [4 5 0 4]]

The 'labels2d' array sorted along axis#0:
 [[4 5 0 2]
 [5 5 3 4]]
The 'labels2d' array sorted along axis#1:
 [[2 3 5 5]
 [0 4 4 5]]

Sorters of the 'labels2d' array along axis#0:
 [[1 0 1 0]
 [0 1 0 1]]
Sorters of the 'labels2d' array along axis#1:
 [[3 2 0 1]
 [2 0 3 1]]

 -- Sorting the 'labels2d' array along axis#1 by indexing it with the sorters:
[[2 3 5 5]
 [0 4 4 5]]


Another reason to sort is to enable faster searching in an array. If the array is sorted, we can use binary search to find elements in logarithmic time.

Searching for $k$ elements in an $n$‑element array with linear search on an unsorted array requires $O(k\cdot n)$ operations. In contrast, sorting the array takes $O(n\cdot \log n)$ time, and then searching for $k$ elements with logarithmic-time search on the sorted array takes $O(k\cdot \log n)$. For sufficiently large $k$ or repeated searches, the latter strategy is ideal.

For **logarithmic-time search** of values `v` in a one‑dimensional sorted array `a`, use **`np.searchsorted(a, v, side, sorter)`**. It is necessary that `a` be a sorted array, or alternatively, we can specify an index array as the `sorter` parameter, which contains the indexing necessary for sorting `a`. The latter can be created with the `np.argsort(a)` command. By definition, `np.searchsorted()` returns the indices in an array with a similar size as `v` such that, when the elements of `v` are inserted into the array `a`, the array `a` remains sorted. In other words, **the `np.searchsorted()` function does not check whether the elements in the `v` array can actually be found in the `a` array**, it only specifies the positions (checked individually) that the elements could have in the `a` array so that it remains sorted. We can check for inclusion using, for example, the `np.isin()` function, which will be discussed later in this chapter, among the set operations.

The `side` parameter of `np.searchsorted()` determines whether, if a searched element in `v` already occurs in the array `a` (even multiple times), it returns the first or last possible index where the searched element could be placed while maintaining the sorted order.


In [ ]:
a = np.array([5,5,3,2,4,5,0,4], dtype=np.int32)
print("The 'a' array:", a)

sa = np.sort(a)
print("The sorted 'sa' array:", sa)

print("\nnp.searchsorted(sa, [-1, 0, 5, 6], side='left'):", np.searchsorted(sa, [-1, 0, 5, 6]))
print("np.searchsorted(sa, [-1, 0, 5, 6], side='right'):", np.searchsorted(sa, [-1, 0, 5, 6], side='right'))

The 'a' array: [5 5 3 2 4 5 0 4]
The sorted 'sa' array: [0 2 3 4 4 5 5 5]

np.searchsorted(sa, [-1, 0, 5, 6], side='left'): [0 0 5 8]
np.searchsorted(sa, [-1, 0, 5, 6], side='right'): [0 1 8 8]


## Finding and counting unique elements: `np.unique()` and `np.bincount()`


NumPy provides several functions for counting distinct elements in arrays and for building histograms. In this chapter we mention two of them.

The `np.unique()` function can be used to collect the unique elements of an array and count them. The first functionality can also be interpreted as producing the set of values contained in the array.


In [ ]:
a = np.array([[99,42,99], [77,42,77]], dtype=np.int32)

print("The 'a' array:\n", a)
print("\na.shape:", a.shape)

print("\nnp.unique(a):", np.unique(a))

The 'a' array:
 [[99 42 99]
 [77 42 77]]

a.shape: (2, 3)

np.unique(a): [42 77 99]


As we can see in the cell above, **calling `np.unique()` on an array without extra parameters returns a *sorted* one‑dimensional array of the distinct elements.**

`np.unique()` also supports an `axis` parameter. When used, the input array is split into slices along the specified axis and the comparison is performed between those slices. For example, if the input is a (2D) matrix, `axis=0` collects the unique rows (sorted), while `axis=1` examines the columns.


In [ ]:
print("The 'a' array:\n", a)

print("\nnp.unique(a, axis=0):\n", np.unique(a, axis=0))

print("\nnp.unique(a, axis=1):\n", np.unique(a, axis=1))

The 'a' array:
 [[99 42 99]
 [77 42 77]]

np.unique(a, axis=0):
 [[77 42 77]
 [99 42 99]]

np.unique(a, axis=1):
 [[42 99]
 [42 77]]


`np.unique()` has three additional optional parameters that enable the following:
- With **`return_index=True`**, besides the array of unique elements/slices, the function returns the **index of the first occurrence** of each unique element/slice. Like `np.argmin()` and `np.argmax()`, the returned indices are flattened indices, i.e. they index the one‑dimensional view of the array.
- With **`return_counts=True`**, the function also returns the **counts** corresponding to each unique element/slice.


In [ ]:
print("The 'a' array:\n", a)

print("\nnp.unique(a, return_index=True):", np.unique(a, return_index=True))

print("\nnp.unique(a, return_counts=True):", np.unique(a, return_counts=True))

print("\nnp.unique(a, return_counts=True):\n", np.unique(a, axis=1, return_counts=True))

The 'a' array:
 [[99 42 99]
 [77 42 77]]

np.unique(a, return_index=True): (array([42, 77, 99], dtype=int32), array([1, 3, 0]))

np.unique(a, return_counts=True): (array([42, 77, 99], dtype=int32), array([2, 2, 2]))

np.unique(a, return_counts=True):
 (array([[42, 99],
       [42, 77]], dtype=int32), array([1, 2]))


- With **`return_inverse=True`**, the function also returns an array that can be used to reconstruct the input array by indexing the array of unique elements.


Besides `np.unique()`, `np.bincount()` can also be used to count unique elements in an array, and in certain cases it can do that much more efficiently. Unlike `np.unique()`, **`np.bincount()` can be used only on one‑dimensional arrays of non‑negative integers.** It does not return the array of unique values; it returns only their counts, in an array of shape `(arrmax+1,)`, where `arrmax` is the maximum element of the input array. The example below illustrates its behavior:


In [ ]:
a = np.array([[7,5,2], [4,5,5]], dtype=np.int32)

print("The 'a' array:\n", a)
print("\na.shape:", a.shape)

print("\nnp.bincount(a):", np.bincount(a.reshape(-1)))

The 'a' array:
 [[7 5 2]
 [4 5 5]]

a.shape: (2, 3)

np.bincount(a): [0 0 1 0 1 3 0 1]


For multidimensional arrays we can use their flattened view as input. As we see above, if the maximum element is 7, we get an array of shape (8,) that stores the frequencies of the values 0, ..., 7.

The functionality provided by `np.bincount()` is much more limited than that of `np.unique()`, but while `np.unique()` needs to check for “new” values as it processes the array, **the implementation of `np.bincount()` is much more efficient**: the output size is determined by a simple maximum search, and the input values can be used directly as indices to increment the corresponding counters. In machine learning, `np.bincount()` is especially useful because, for example in classification, labels are typically represented as integers in the interval [0, n_labels−1].

If we know the maximum element in the input array in advance, or want to obtain an array of fixed size, `np.bincount()` can also be used with the `minlength` parameter:

In [ ]:
print("np.bincount(a):", np.bincount(a.reshape(-1), minlength=20))

np.bincount(a): [0 0 1 0 1 3 0 1 0 0 0 0 0 0 0 0 0 0 0 0]


Naturally, if the values in the input array are very sparse in the interval [0, arrmax+1], - for example, if the array contains very large values relative to its length - then `np.bincount()` is no longer efficient, because it would require allocating an excessively large output array.


## Set operations


The Python `set` container is designed specifically to represent mathematical sets: it enforces element uniqueness by design and therefore (via hashing) enables faster‑than‑linear lookup. NumPy does not provide a dedicated set container, but we can represent sets with `ndarray` arrays and perform set operations on them using a few dedicated NumPy functions. If we are not sure whether our array contains duplicate values, we can use `np.unique()` to filter out duplicates.


In [ ]:
a = np.array([5,5,3,2,4,5,0,4], dtype=np.int32)
u = np.unique(a)

print("The 'a' array:", a)
print("The np.unique(a) array:", u)      # np.unique() sorts items in the returned array

The 'a' array: [5 5 3 2 4 5 0 4]
The np.unique(a) array: [0 2 3 4 5]


The basic set operations are implemented by the following NumPy functions:
- Membership (is element in set): `np.isin()`
- Union: `np.union1d()`
- Intersection: `np.intersect1d()`
- Set difference: `np.setdiff1d()`
- Symmetric difference (elements in exactly one set): `np.setxor1d()`

Most of these functions have an `assume_unique` parameter. If set to true, the function assumes there are no duplicates in the input arrays, which can enable faster execution. By default this parameter is false, so NumPy’s set operations do not require that the arrays contain only unique elements.

The functions above **operate on one‑dimensional arrays**, so they are not directly suitable for sets of vectors. `np.isin(element, test_elements)` is an exception in the sense that the array of elements (`element`) that we search for individually in the other array (`test_elements`) can be multidimensional. The logical mask we get back will have the same shape as the `element` array and will indicate for each of its elements whether it is contained in the `test_elements` array.


In [ ]:
a1 = np.array([5,5,3,2,4,5,0,4], dtype=np.int32)
a2 = np.array([1,1,0,4,1,6], dtype=np.int32)

print("The 'a1' array:", a1)
print("The 'a2' array:", a2)

print("\nnp.isin(a1, a2):", np.isin(a1, a2))    # whether items in a1 are in a2, elementwise
print("np.union1d(a1, a2):", np.union1d(a1, a2))
print("np.intersect1d(a1, a2):", np.intersect1d(a1, a2))
print("np.setdiff1d(a1, a2):", np.setdiff1d(a1, a2))
print("np.setxor1d(a1, a2):", np.setxor1d(a1, a2))

The 'a1' array: [5 5 3 2 4 5 0 4]
The 'a2' array: [1 1 0 4 1 6]

np.isin(a1, a2): [False False False False  True False  True  True]
np.union1d(a1, a2): [0 1 2 3 4 5 6]
np.intersect1d(a1, a2): [0 4]
np.setdiff1d(a1, a2): [2 3 5]
np.setxor1d(a1, a2): [1 2 3 5 6]


## Other NumPy functions


We list a few additional important functions in this chapter.

The call **`np.clip(arr, arr_max, arr_min)`** is equivalent to `np.minimum(arr_max, np.maximum(arr, arr_min))`, but is more efficient. It “clips” the elements of an array between `arr_min` and `arr_max` by **replacing elements smaller than `arr_min` with `arr_min`, and elements larger than `arr_max` with `arr_max`.** `arr_min` and `arr_max` do not have to be constants; they can also be arrays with shapes broadcastable against `arr`.


In [ ]:
a = np.array([[-7.5,5.2,2.5], [-4.2,5.9,9.5]], dtype=np.float32)

print("The 'a' array:\n", a)
print("\na.shape:", a.shape)

print("\nnp.clip(a, -5, 5):\n", np.clip(a, -5, 5))

# rows
print("\nnp.clip(a, [-5, -4, -3], [5, 4, 3]):\n", np.clip(a, [-5, -4, -3], [5, 4, 3]))  # [5, 4, 3] list is converted to (3,)
                                                                                        #     shaped array which is broadcasted to (2,3)

# columns
print("\nnp.clip(a, [[-5],[-3]], [[5],[3]]):\n", np.clip(a, [[-5],[-3]], [[5],[3]]))   # [[5], [3]] nested lists are converted to (2,1)
                                                                                       #     shaped array which is broadcasted to (2,3)

The 'a' array:
 [[-7.5  5.2  2.5]
 [-4.2  5.9  9.5]]

a.shape: (2, 3)

np.clip(a, -5, 5):
 [[-5.   5.   2.5]
 [-4.2  5.   5. ]]

np.clip(a, [-5, -4, -3], [5, 4, 3]):
 [[-5.          4.          2.5       ]
 [-4.19999981  4.          3.        ]]

np.clip(a, [[-5],[-3]], [[5],[3]]):
 [[-5.   5.   2.5]
 [-3.   3.   3. ]]


Additional elementwise operations:
- Absolute value: `np.fabs()`
- Sign (signum) function: `np.sign()`
- Squaring: `np.square()`
- Square root: `np.sqrt()`


In [ ]:
a = np.arange(-3, 3, dtype=np.float32)
print("The 'a' array:", a)

print("\nnp.fabs(a):", np.fabs(a))
print("np.sign(a):", np.sign(a))
print("np.square(a):", np.square(a))
print("np.sqrt(a):", np.sqrt(a))   # sqrt(negative) is NaN; RuntimeWarning: invalid value encountered in sqrt

The 'a' array: [-3. -2. -1.  0.  1.  2.]

np.fabs(a): [3. 2. 1. 0. 1. 2.]
np.sign(a): [-1. -1. -1.  0.  1.  1.]
np.square(a): [9. 4. 1. 0. 1. 4.]
np.sqrt(a): [      nan       nan       nan 0.        1.        1.4142135]


<ipython-input-35-be6e61239115>:7: RuntimeWarning: invalid value encountered in sqrt
  print("np.sqrt(a):", np.sqrt(a))   # sqrt(negative) is NaN + RuntimeWarning: invalid value encountered in sqrt


Mathematical operations can produce special constants in extreme or undefined cases when working with floating‑point types. Examples include NaN (Not‑a‑Number), Inf (infinity), and −Inf (minus infinity).

Additional mathematical functions for elementwise operations:
- Trigonometry: `np.sin()`, `np.cos()`, `np.tan()`
- Logarithms (bases e, 10 and 2): `np.log()`, `np.log10()`, `np.log2()`
- Exponential: `np.exp()`


In [ ]:
a = np.arange(-3, 3, dtype=np.float32)
print("The 'a' array:", a)

print("\nnp.sin(a):", np.sin(a))
print("np.cos(a):", np.cos(a))
print("np.tan(a):", np.tan(a))
print("np.log(a):", np.log(a))    # RuntimeWarning: invalid value encountered in log, RuntimeWarning: divide by zero encountered in log
print("np.log10(a):", np.log10(a)) # RuntimeWarning: invalid value encountered in log10, RuntimeWarning: divide by zero encountered in log10
print("np.log2(a):", np.log2(a))  # RuntimeWarning: invalid value encountered in log2, RuntimeWarning: divide by zero encountered in log2
print("np.exp(a):", np.exp(a))

The 'a' array: [-3. -2. -1.  0.  1.  2.]

np.sin(a): [-0.14112   -0.9092974 -0.841471   0.         0.841471   0.9092974]
np.cos(a): [-0.9899925 -0.4161468  0.5403023  1.         0.5403023 -0.4161468]
np.tan(a): [ 0.14254655  2.1850398  -1.5574077   0.          1.5574077  -2.1850398 ]
np.log(a): [      nan       nan       nan      -inf 0.        0.6931472]
np.log10(a): [    nan     nan     nan    -inf 0.      0.30103]
np.log2(a): [ nan  nan  nan -inf   0.   1.]
np.exp(a): [0.04978707 0.13533528 0.36787942 1.         2.718282   7.3890557 ]


<ipython-input-36-8952a70decc7>:7: RuntimeWarning: divide by zero encountered in log
  print("np.log(a):", np.log(a))    # RuntimeWarning: invalid value encountered in log, RuntimeWarning: divide by zero encountered in log
<ipython-input-36-8952a70decc7>:7: RuntimeWarning: invalid value encountered in log
  print("np.log(a):", np.log(a))    # RuntimeWarning: invalid value encountered in log, RuntimeWarning: divide by zero encountered in log
<ipython-input-36-8952a70decc7>:8: RuntimeWarning: divide by zero encountered in log10
  print("np.log10(a):", np.log10(a)) # RuntimeWarning: invalid value encountered in log10, RuntimeWarning: divide by zero encountered in log10
<ipython-input-36-8952a70decc7>:8: RuntimeWarning: invalid value encountered in log10
  print("np.log10(a):", np.log10(a)) # RuntimeWarning: invalid value encountered in log10, RuntimeWarning: divide by zero encountered in log10
<ipython-input-36-8952a70decc7>:9: RuntimeWarning: divide by zero encountered in log2
  print("n

- Conversion between angles measured in degrees and radians: `np.deg2rad()`, `np.rad2deg()`


In [ ]:
a = np.array([-720, 0, 45, 180, 360, 720], dtype=np.float32)
print("The 'a' array:", a)

print("\nnp.deg2rad(a):", np.deg2rad(a))
print("np.rad2deg(a):", np.rad2deg(a))

The 'a' array: [-720.    0.   45.  180.  360.  720.]

np.deg2rad(a): [-12.566371    0.          0.7853982   3.1415927   6.2831855  12.566371 ]
np.rad2deg(a): [-41252.96      0.     2578.31  10313.24  20626.48  41252.96]


- Cumulative sum: `np.cumsum()`
- Cumulative product: `np.cumprod()`
- Difference between neighboring elements: `np.diff()`. With parameter `n` we can specify how many times to apply the difference operation repeatedly. `np.diff(a)` is equivalent to `a[:-1] - a[1:]`.

All three functions support the `axis` parameter.


In [ ]:
a = np.array([7,-4,8,5,2,-2,3.5], dtype=np.float32)
print("The 'a' array:", a)

print("\nnp.cumsum(a):", np.cumsum(a))
print("np.cumprod(a):", np.cumprod(a))
print("\nnp.diff(a):", np.diff(a))
print("a[:-1] - a[1:]:", a[:-1] - a[1:])
print("np.diff(a, diff=2):", np.diff(a, n=2))
print("np.diff(a[:-1] - a[1:]):", np.diff(a[:-1] - a[1:]))

The 'a' array: [ 7.  -4.   8.   5.   2.  -2.   3.5]

np.cumsum(a): [ 7.   3.  11.  16.  18.  16.  19.5]
np.cumprod(a): [ 7.000e+00 -2.800e+01 -2.240e+02 -1.120e+03 -2.240e+03  4.480e+03
  1.568e+04]

np.diff(a): [-11.   12.   -3.   -3.   -4.    5.5]
a[:-1] - a[1:]: [ 11.  -12.    3.    3.    4.   -5.5]
np.diff(a, diff=2): [ 23.  -15.    0.   -1.    9.5]
np.diff(a[:-1] - a[1:]): [-23.   15.    0.    1.   -9.5]
